In [1]:
import sys
from pathlib import Path

PROJECT_DIR = Path('.').resolve().parent
if str(PROJECT_DIR) not in sys.path: sys.path.insert(0, str(PROJECT_DIR))

import numpy as np
# from sklearn.metrics import confusion_matrix
# from sklearn.metrics import roc_auc_score

from collections import defaultdict



import networks.ParT_ABCDiscoTEC_split as ParT
import user_scripts.preprocess as preprocess
from   user_scripts.branches_to_get import get_branchDict
import user_scripts.val_plots2 as val_plots
import utils.network_helpers as nh

from utils.vtxLevelDataset import ModifiedUprootIterator
from utils.help_preprocess import probe_shapes
from utils.optimizers.ranger import Ranger

import matplotlib.pyplot as plt


import torch
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR

import user_scripts.ABCDiscoTEC_loss as ABCD

import neptune
from neptune.utils import stringify_unsupported

import datetime
from functools import partial
import glob
import gc
import math
import warnings
import os
import random
import json
import copy


DATA_READ_BASEPATH  = '/scratch-cbe/users/alikaan.gueven/ML_KAAN'
RUN_SAVE_BASEPATH   = '/groups/hephy/cms/alikaan.gueven/ParT/runs'
MODEL_SAVE_BASEPATH = '/groups/hephy/cms/alikaan.gueven/ParT/models'

glob_dirs = [os.path.join(DATA_READ_BASEPATH, 'CustomNanoAOD_MLtraining_20250910_mixed')]

tmpSigList = []
for sample_dir in glob_dirs:
    tmpSigList.extend(glob.glob(f'{sample_dir}/**/*.root', recursive=True))

tmpSigList.sort()            # make deterministic order
random.seed(42)              # set reproducible seed
random.shuffle(tmpSigList)   # shuffle in reproducible way

tmpSigList = [sig + ':Events' for sig in tmpSigList]


minTrain = round(len(tmpSigList)*0.00)
maxTrain = round(len(tmpSigList)*0.80)

trainSigList = tmpSigList[minTrain:maxTrain]


trainDict = {
    'sig': trainSigList,
    'bkg': None
}


branchDict_dataset    = get_branchDict()

shuffle = False
nWorkers = 4
step_size = 4000

trainDataset = ModifiedUprootIterator(trainDict,
                                      branchDict_dataset,
                                      shuffle=shuffle,
                                      nWorkers=nWorkers,
                                      step_size=step_size)

X = next(trainDataset)
X['SDVTrack_pt']

prefetch_factor = 16

branchDict_dataloader = copy.deepcopy(branchDict_dataset)
branchDict_dataloader['ev'].append('event_idx')

preprocess_fn = partial(preprocess.transform, branch_dict=branchDict_dataloader)


trainLoader = torch.utils.data.DataLoader(trainDataset, 
                                          num_workers=nWorkers,
                                          prefetch_factor=prefetch_factor,
                                          persistent_workers= True,
                                          collate_fn=preprocess_fn,
                                          drop_last=True, 
                                          pin_memory=True)

for batch_num, X in enumerate(trainLoader):
    df = ABCD.select_leading_vertices(X['sv_features'][:,0,0],
                                      X['sv_features'][:,1,0],
                                      X['event_idx'],
                                      X['label'])
    print(df[-1].shape, df[0][:5])
    if batch_num > 50:
        break

/groups/hephy/cms/alikaan.gueven/conda/envs/SDV/lib/python3.11/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
cling::DynamicLibraryManager::loadLibrary(): libGL.so.1: cannot open shared object file: No such file or directory
Error in <AutoloadLibraryMU>: Failed to load library /groups/hephy/cms/alikaan.gueven/conda/envs/SDV/lib/libEve.so.6.28.00cling JIT session error: Failed to materialize symbols: { (main, { _ZeqRK7TStringRKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE }) }


Welcome to JupyROOT 6.28/00


Initialize iterable dataset
nWorkers:  4


<Array [[0.904, 1.51, ..., 1.38, 0.586], ...] type='4000 * var * float32'>

In [24]:
# X['event_idx']

In [49]:
prefetch_factor = 16

branchDict_dataloader = copy.deepcopy(branchDict_dataset)
branchDict_dataloader['ev'].append('event_idx')

preprocess_fn = partial(preprocess.transform, branch_dict=branchDict_dataloader)


trainLoader = torch.utils.data.DataLoader(trainDataset, 
                                          num_workers=nWorkers,
                                          prefetch_factor=prefetch_factor,
                                          persistent_workers= True,
                                          collate_fn=preprocess_fn,
                                          drop_last=True, 
                                          pin_memory=True)

In [48]:
for batch_num, X in enumerate(trainLoader):
    df = ABCD.select_leading_vertices(X['sv_features'][:,0,0],
                                      X['sv_features'][:,1,0],
                                      X['event_idx'],
                                      X['label'])
    print(df[-1].shape, df[0][:5])
    if batch_num > 50:
        break

__iter__ is called.
torch.Size([2866]) tensor([5.3462, 5.4190, 4.9791, 5.4331, 4.8957], dtype=torch.float64)
torch.Size([2980]) tensor([6.3276, 5.3185, 5.3179, 5.2191, 5.7001], dtype=torch.float64)
torch.Size([2900]) tensor([5.3693, 5.2909, 5.2201, 5.0766, 5.7667], dtype=torch.float64)
torch.Size([2877]) tensor([5.0580, 5.6126, 5.2742, 4.9250, 5.4869], dtype=torch.float64)
torch.Size([2889]) tensor([5.4958, 5.3189, 5.5137, 5.3115, 5.6521], dtype=torch.float64)
torch.Size([3036]) tensor([5.4969, 5.1310, 5.0677, 5.3005, 5.1551], dtype=torch.float64)
torch.Size([2909]) tensor([5.2280, 5.6121, 5.1926, 5.0629, 5.1638], dtype=torch.float64)
torch.Size([3164]) tensor([5.0967, 5.3897, 5.2277, 5.0387, 5.1154], dtype=torch.float64)


KeyboardInterrupt: 

In [30]:
X['event_idx']

tensor([[76000],
        [76000],
        [76001],
        ...,
        [79997],
        [79998],
        [79999]])

In [32]:
a, b = torch.unique(X['event_idx'], return_inverse=True)

In [9]:
X['event_idx'].shape

torch.Size([5298, 1])

In [10]:
X['label'].shape

torch.Size([5298, 1])

In [11]:
X['sv_features'].shape

torch.Size([5298, 9, 1])

In [21]:
df[-1].shape

torch.Size([2866])

In [14]:
X['sv_features'].shape

torch.Size([5298, 9, 1])

In [ ]:
X['label'].shape

In [ ]:
X['event_idx'].shape

In [ ]:
X['tk_features'].shape

In [ ]:
A = trainDataset.__next__()

In [ ]:
A['MET_phi'].type

In [ ]:
A['SDVSecVtx_pt'].type

In [ ]:
import utils.help_preprocess as hp


In [ ]:
branchDict = get_branchDict()
df = hp.pad_and_fill(A, branchDict)

In [ ]:
df['MET_phi'].shape

In [ ]:
df['SDVSecVtx_pt'].shape

In [ ]:
def pad_and_fill_ev(X, branchDict, svDim=12, tkDim=10, fillValue=-9e10):
    def process_field(field, to_broadcast=False):
        field_fillValue = {
                'SDVSecVtx_E': 1e3,
                'SDVSecVtx_pz': 0
            }.get(field, fillValue)
        if to_broadcast:
            filled = X[field]
        else:
            padded = ak.pad_none(X[field], target=svDim, clip=True, axis=-1)
            filled = ak.fill_none(padded, field_fillValue, axis=-1)
        
        X_np = filled.to_numpy()
        return torch.tensor(X_np)

    X_dict = {}
    for field in X.fields:
        if branchDict['ev'] and field in branchDict['ev']:
            if field.startswith('n'):
                X[field] = ak.values_astype(X[field], np.int32)
            elif field.startswith('Jet'):
                X[field] = X[field][:, 0]
            
            X_dict[field] = process_field(field, to_broadcast=True)
        
        elif branchDict['sv'] and field in branchDict['sv']:
            X_dict[field] = process_field(field)
        
        elif branchDict['label'] and field in branchDict['label']:
            X_dict[field] = process_field(field)

        elif branchDict['tk'] and field in branchDict['tk']:
            trIdx = X.SDVIdxLUT_TrackIdx
            svIdx = X.SDVIdxLUT_SecVtxIdx
            n_sv = X.nSDVSecVtx
            
            builder = ak.ArrayBuilder()
            deepTable(X[field], trIdx, svIdx, n_sv, builder)
            deepX = builder.snapshot()

            field_fillValue = {
                'SDVTrack_E': 1e3,
                'SDVTrack_pz': 0
            }.get(field, fillValue)
            
            padded = ak.pad_none(deepX, target=tkDim, clip=True, axis=2)
            filled = ak.fill_none(padded, field_fillValue, axis=2)
            
            padded = ak.pad_none(filled, target=svDim, clip=True, axis=1)
            filled = ak.fill_none(padded, [field_fillValue]*tkDim, axis=1)

            
            
            X_np = filled.to_numpy()
            X_tensor = torch.tensor(X_np) 
            X_dict[field] = X_tensor

    return X_dict

In [ ]:
import awkward as ak
from utils.help_preprocess import *

branchDict = get_branchDict()
branchDict['ev'].append('event_idx')


df2 = pad_and_fill_ev(A, branchDict)

In [ ]:
df2['SDVTrack_pt'].shape

In [ ]:
ev_idx = torch.tensor([0, 1, 1, 2, 2, 2])
scores = torch.tensor([0.7, 0.5, 0.6, 0.3, 0.5, 0.1])

max_scores = torch.full((3,), float("-inf"))
max_scores.scatter_reduce_(
        dim=0,
        index=ev_idx,
        src=scores,
        reduce="amax",
        include_self=True,
)

In [ ]:
max_scores[ev_idx]